# <h2 style='color:blue' align='center'>**Chest X-Ray Pneumonia Classification: CNN vs ResNet50** 🫁</h2>

### **CAP 4630 - Intro to Artificial Intelligence | Final Project**

**Team Member (Training & Validation):** Christina Pappachan

**Dataset:** [Chest X-Ray Pneumonia Balanced Dataset](https://www.kaggle.com/datasets/yusufmurtaza01/chest-xray-pneumonia-balanced-dataset/data)

**Split:** 80% Train | 10% Validation | 10% Test (test handled by teammate) 🤝

<div style="border-radius:12px; padding: 20px; background-color: #e2c9ff; font-size:115%; text-align:left">

<h2 align="left"><font color=#8502d1>About the Project</font></h2>

We classify chest X-ray images into two classes:

- **NORMAL** 🟢 — Healthy lungs
- **PNEUMONIA** 🔴 — Lungs showing pneumonia infection

We compare **two approaches**:

1. **Model 1: Custom CNN built from scratch** — Like Coding Session 6 (CIFAR-10), Sequential CNN with Conv2D + MaxPooling layers. All weights start as random numbers and learn from scratch. 🧠

2. **Model 2: ResNet50 with Transfer Learning** — A deep CNN pre-trained on ImageNet (millions of images). We replace the final layer and fine-tune it for chest X-rays. Like Coding Session 7 used pre-trained models. 🏗️

At the end, we compare both models side-by-side using accuracy, precision, recall, F1, and confusion matrices.

</div>

<h4 style="color:purple">Step 1: Mount Google Drive 📂</h4>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


<h4 style="color:purple">Step 2: Unzip the Dataset (One-Time Setup) 📦</h4>

Update `zip_path` below to point to the dataset zip file in your Drive. This cell unzips the dataset into Colab's local storage (`/content/`) which is way faster than reading directly from Drive during training. ⚡

**Note:** Colab's local storage gets wiped when the runtime disconnects, so you'll need to re-run this cell each new session. The zip stays in your Drive though, so it only takes a minute to extract.

In [ ]:
import os
import zipfile

# 👉 UPDATE THIS PATH to where your dataset zip lives in Drive
zip_path = '/content/drive/MyDrive/final_dataset_80_10_10.zip'

# Where to extract the dataset (Colab local storage = fast)
extract_path = '/content/dataset'

# Skip extraction if already done in this runtime
if os.path.exists(extract_path) and os.listdir(extract_path):
    print(f'✅ Dataset already extracted at {extract_path}')
else:
    print(f'📦 Extracting {zip_path}...')
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print(f'✅ Extracted to {extract_path}')

# Show what was extracted (top-level only)
print('\n📁 Top-level contents:')
for item in sorted(os.listdir(extract_path)):
    full_path = os.path.join(extract_path, item)
    if os.path.isdir(full_path):
        print(f'  📁 {item}/')
    else:
        print(f'  📄 {item}')

📦 Extracting /content/drive/MyDrive/final_dataset_80_10_10.zip...
✅ Extracted to /content/dataset

📁 Top-level contents:
  📁 __MACOSX/
  📁 final_dataset_80_10_10/


<h4 style="color:purple">Step 3: Locate Train and Val Folders 🗂️</h4>

After extraction, your folder structure should look like this:

```
/content/dataset/...
├── train/
│   ├── NORMAL/
│   └── PNEUMONIA/
├── val/
│   ├── NORMAL/
│   └── PNEUMONIA/
└── test/
    ├── NORMAL/
    └── PNEUMONIA/
```

Sometimes zips have a wrapper folder (like `chest_xray/train/...`). The cell below auto-detects where `train` and `val` actually live. 🔍

In [ ]:
# Auto-detect train and val folders (in case there's a wrapper folder)
def find_folder(root, target):
    for dirpath, dirnames, _ in os.walk(root):
        if target in dirnames:
            return os.path.join(dirpath, target)
    return None

train_dir = find_folder(extract_path, 'train')
val_dir   = find_folder(extract_path, 'val')

# Some datasets use 'valid' instead of 'val'
if val_dir is None:
    val_dir = find_folder(extract_path, 'valid')

print(f'📁 train_dir: {train_dir}')
print(f'📁 val_dir:   {val_dir}')

if train_dir is None or val_dir is None:
    print('\n⚠️ Could not find train/val folders. Run this to inspect the structure:')
    print(f'   !find {extract_path} -maxdepth 3 -type d')
else:
    print('\n✅ Found both folders!')
    print(f'\nTrain classes: {sorted(os.listdir(train_dir))}')
    print(f'Val classes:   {sorted(os.listdir(val_dir))}')

    for cls in sorted(os.listdir(train_dir)):
        print(f'  Train/{cls}: {len(os.listdir(os.path.join(train_dir, cls)))} images')
    for cls in sorted(os.listdir(val_dir)):
        print(f'  Val/{cls}:   {len(os.listdir(os.path.join(val_dir, cls)))} images')

📁 train_dir: /content/dataset/__MACOSX/final_dataset_80_10_10/train
📁 val_dir:   /content/dataset/__MACOSX/final_dataset_80_10_10/val

✅ Found both folders!

Train classes: ['._NORMAL', '._PNEUMONIA', 'NORMAL', 'PNEUMONIA']
Val classes:   ['._NORMAL', '._PNEUMONIA', 'NORMAL', 'PNEUMONIA']


NotADirectoryError: [Errno 20] Not a directory: '/content/dataset/__MACOSX/final_dataset_80_10_10/train/._NORMAL'

<h4 style="color:purple">Step 4: Import Libraries 📦</h4>

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pandas as pd

# Check if GPU is available
print('GPU Available:', tf.config.list_physical_devices('GPU'))

<h4 style="color:purple">Step 5: Set Image Parameters ⚙️</h4>

In [ ]:
IMG_HEIGHT = 224   # ResNet50's expected input size
IMG_WIDTH  = 224
BATCH_SIZE = 32
EPOCHS     = 10

classes = ['NORMAL', 'PNEUMONIA']

# <h2 style='color:#1976d2'>**Part 1: Custom CNN from Scratch** 🧠</h2>

<h4 style="color:purple">Step 6: Load and Normalize the Images 🎨</h4>

Just like in Coding Session 6, we normalize images to a 0–1 range by dividing by 255. The training generator also adds light data augmentation to help the model generalize.

In [ ]:
# Training data: normalize + light augmentation 🌀
train_datagen_cnn = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    zoom_range=0.1,
    horizontal_flip=True
)

# Validation data: just normalize, no augmentation
val_datagen_cnn = ImageDataGenerator(rescale=1./255)

train_generator_cnn = train_datagen_cnn.flow_from_directory(
    train_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    classes=classes,
    shuffle=True
)

val_generator_cnn = val_datagen_cnn.flow_from_directory(
    val_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    classes=classes,
    shuffle=False
)

print('\nClass indices:', train_generator_cnn.class_indices)

<h4 style="color:purple">Step 7: Visualize Some Sample Images 🖼️</h4>

In [ ]:
X_batch, y_batch = next(train_generator_cnn)

plt.figure(figsize=(15, 15))
for i in range(16):
    plt.subplot(4, 4, i+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(X_batch[i])
    plt.title(classes[int(y_batch[i])], fontsize=12)
plt.show()

train_generator_cnn.reset()

<h4 style="color:purple">Step 8: Build the Custom CNN 🧠</h4>

Same structure as Coding Session 6 — Conv2D + MaxPooling layers to extract features, then Flatten + Dense for classification. We add one extra Conv2D block since these X-rays are bigger (224x224) than CIFAR-10's tiny 32x32 images, plus a Dropout layer to fight overfitting.

In [ ]:
cnn = models.Sequential([
    layers.Conv2D(filters=32, kernel_size=(3, 3), activation='relu', input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(filters=64, kernel_size=(3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(filters=128, kernel_size=(3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

cnn.summary()

<h4 style="color:purple">Step 9: Compile the Custom CNN ⚙️</h4>

Since this is binary classification (NORMAL vs PNEUMONIA), we use `binary_crossentropy`. Optimizer stays as `adam` like in class. ✨

In [ ]:
cnn.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

<h4 style="color:purple">Step 10: Train the Custom CNN 🏋️</h4>

In [ ]:
history_cnn = cnn.fit(
    train_generator_cnn,
    validation_data=val_generator_cnn,
    epochs=EPOCHS
)

<h4 style="color:purple">Step 11: Plot Custom CNN Training History 📈</h4>

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_cnn.history['accuracy'], label='Training Accuracy')
axes[0].plot(history_cnn.history['val_accuracy'], label='Validation Accuracy')
axes[0].set_title('Custom CNN - Accuracy over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history_cnn.history['loss'], label='Training Loss')
axes[1].plot(history_cnn.history['val_loss'], label='Validation Loss')
axes[1].set_title('Custom CNN - Loss over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

<h4 style="color:purple">Step 12: Evaluate Custom CNN on Validation ✅</h4>

In [ ]:
cnn_val_loss, cnn_val_accuracy = cnn.evaluate(val_generator_cnn)
print(f'\n🎯 Custom CNN Validation Loss:     {cnn_val_loss:.4f}')
print(f'🎯 Custom CNN Validation Accuracy: {cnn_val_accuracy:.4f}')

val_generator_cnn.reset()
y_pred_probs_cnn = cnn.predict(val_generator_cnn)
y_pred_classes_cnn = [1 if prob > 0.5 else 0 for prob in y_pred_probs_cnn]
y_true = val_generator_cnn.classes

print('\nClassification Report (Custom CNN):\n')
print(classification_report(y_true, y_pred_classes_cnn, target_names=classes))

cm_cnn = confusion_matrix(y_true, y_pred_classes_cnn)
plt.figure(figsize=(6, 5))
sns.heatmap(cm_cnn, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Custom CNN - Confusion Matrix')
plt.show()

<h4 style="color:purple">Step 13: Save the Custom CNN 💾</h4>

In [ ]:
cnn.save('/content/drive/MyDrive/pneumonia_custom_cnn.h5')
cnn.save('pneumonia_custom_cnn.h5')
print('✅ Custom CNN saved!')

# <h2 style='color:#1976d2'>**Part 2: ResNet50 with Transfer Learning** 🏗️</h2>

<div style="border-radius:12px; padding: 20px; background-color: #fff3cd; font-size:110%; text-align:left">

<h3 align="left"><font color=#856404>Why ResNet50? 🤔</font></h3>

ResNet50 is a deep CNN with **50 layers** that has already been trained on millions of ImageNet images. Its key innovation is **residual (skip) connections** that solve the vanishing gradient problem in very deep networks.

**Transfer learning approach:**
1. Load ResNet50 with pre-trained ImageNet weights (no top layer)
2. Freeze the base layers so we don't lose what they learned
3. Add our own classification head (Dense → Dropout → Dense)
4. Train only the new layers on chest X-rays

This way we leverage features ResNet50 already learned (edges, textures, shapes) and just teach it to apply them to medical imaging. ✨

</div>

<h4 style="color:purple">Step 14: Set Up Generators for ResNet50 🎨</h4>

ResNet50 expects images normalized in a specific way (different from just dividing by 255), so we use its built-in `preprocess_input` function.

In [ ]:
train_datagen_resnet = ImageDataGenerator(
    preprocessing_function=resnet_preprocess,
    rotation_range=10,
    zoom_range=0.1,
    horizontal_flip=True
)

val_datagen_resnet = ImageDataGenerator(preprocessing_function=resnet_preprocess)

train_generator_resnet = train_datagen_resnet.flow_from_directory(
    train_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    classes=classes,
    shuffle=True
)

val_generator_resnet = val_datagen_resnet.flow_from_directory(
    val_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    classes=classes,
    shuffle=False
)

<h4 style="color:purple">Step 15: Build the ResNet50 Transfer Learning Model 🏗️</h4>

In [ ]:
# Load ResNet50 pre-trained on ImageNet (without the top classification layer)
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)
)

# Freeze the base model so its weights don't change during training
base_model.trainable = False

# Build the full model: ResNet50 base + our custom classification head
resnet_model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

resnet_model.summary()

<h4 style="color:purple">Step 16: Compile the ResNet50 Model ⚙️</h4>

In [ ]:
resnet_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

<h4 style="color:purple">Step 17: Train the ResNet50 Model 🏋️</h4>

In [ ]:
history_resnet = resnet_model.fit(
    train_generator_resnet,
    validation_data=val_generator_resnet,
    epochs=EPOCHS
)

<h4 style="color:purple">Step 18: Plot ResNet50 Training History 📈</h4>

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_resnet.history['accuracy'], label='Training Accuracy')
axes[0].plot(history_resnet.history['val_accuracy'], label='Validation Accuracy')
axes[0].set_title('ResNet50 - Accuracy over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history_resnet.history['loss'], label='Training Loss')
axes[1].plot(history_resnet.history['val_loss'], label='Validation Loss')
axes[1].set_title('ResNet50 - Loss over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

<h4 style="color:purple">Step 19: Evaluate ResNet50 on Validation ✅</h4>

In [ ]:
resnet_val_loss, resnet_val_accuracy = resnet_model.evaluate(val_generator_resnet)
print(f'\n🎯 ResNet50 Validation Loss:     {resnet_val_loss:.4f}')
print(f'🎯 ResNet50 Validation Accuracy: {resnet_val_accuracy:.4f}')

val_generator_resnet.reset()
y_pred_probs_resnet = resnet_model.predict(val_generator_resnet)
y_pred_classes_resnet = [1 if prob > 0.5 else 0 for prob in y_pred_probs_resnet]
y_true = val_generator_resnet.classes

print('\nClassification Report (ResNet50):\n')
print(classification_report(y_true, y_pred_classes_resnet, target_names=classes))

cm_resnet = confusion_matrix(y_true, y_pred_classes_resnet)
plt.figure(figsize=(6, 5))
sns.heatmap(cm_resnet, annot=True, fmt='d', cmap='Greens',
            xticklabels=classes, yticklabels=classes)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('ResNet50 - Confusion Matrix')
plt.show()

<h4 style="color:purple">Step 20: Save the ResNet50 Model 💾</h4>

In [ ]:
resnet_model.save('/content/drive/MyDrive/pneumonia_resnet50.h5')
resnet_model.save('pneumonia_resnet50.h5')
print('✅ ResNet50 model saved!')

# <h2 style='color:#d32f2f'>**Part 3: Side-by-Side Comparison** ⚔️</h2>

<h4 style="color:purple">Step 21: Compare Validation Metrics 📊</h4>

In [ ]:
results = {
    'Model': ['Custom CNN (from scratch)', 'ResNet50 (transfer learning)'],
    'Accuracy':  [accuracy_score(y_true, y_pred_classes_cnn),  accuracy_score(y_true, y_pred_classes_resnet)],
    'Precision': [precision_score(y_true, y_pred_classes_cnn), precision_score(y_true, y_pred_classes_resnet)],
    'Recall':    [recall_score(y_true, y_pred_classes_cnn),    recall_score(y_true, y_pred_classes_resnet)],
    'F1 Score':  [f1_score(y_true, y_pred_classes_cnn),        f1_score(y_true, y_pred_classes_resnet)]
}

results_df = pd.DataFrame(results)
print('\n📊 MODEL COMPARISON 📊\n')
print(results_df.to_string(index=False))

results_df.to_csv('model_comparison.csv', index=False)

<h4 style="color:purple">Step 22: Bar Chart Comparison 📈</h4>

In [ ]:
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
cnn_scores    = [results[m][0] for m in metrics]
resnet_scores = [results[m][1] for m in metrics]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, cnn_scores,    width, label='Custom CNN',  color='#1976d2')
bars2 = ax.bar(x + width/2, resnet_scores, width, label='ResNet50',    color='#388e3c')

ax.set_xlabel('Metric')
ax.set_ylabel('Score')
ax.set_title('Custom CNN vs ResNet50 - Validation Performance')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(axis='y', alpha=0.3)

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{height:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

<h4 style="color:purple">Step 23: Side-by-Side Confusion Matrices 🧮</h4>

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm_cnn, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes, ax=axes[0])
axes[0].set_title('Custom CNN - Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

sns.heatmap(cm_resnet, annot=True, fmt='d', cmap='Greens',
            xticklabels=classes, yticklabels=classes, ax=axes[1])
axes[1].set_title('ResNet50 - Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

<h4 style="color:purple">Step 24: Training History Comparison 📉</h4>

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_cnn.history['val_accuracy'],    label='Custom CNN', color='#1976d2')
axes[0].plot(history_resnet.history['val_accuracy'], label='ResNet50',   color='#388e3c')
axes[0].set_title('Validation Accuracy Comparison')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history_cnn.history['val_loss'],    label='Custom CNN', color='#1976d2')
axes[1].plot(history_resnet.history['val_loss'], label='ResNet50',   color='#388e3c')
axes[1].set_title('Validation Loss Comparison')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

<h4 style="color:purple">Step 25: Download the Saved Models 📥</h4>

In [ ]:
from google.colab import files
files.download('pneumonia_custom_cnn.h5')
files.download('pneumonia_resnet50.h5')

In [ ]:
# ============================================================
# 🧪 PART 4: TESTING ON UNSEEN TEST SET 🧪
# ============================================================
import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, precision_score, recall_score, f1_score)

# --- Mount Google Drive ---
drive.mount('/content/drive')

# --- Settings (in case running this cell standalone) ---
IMG_HEIGHT = 224
IMG_WIDTH  = 224
BATCH_SIZE = 32
classes = ['NORMAL', 'PNEUMONIA']

# 👉 Paths to your saved models in Drive
MODEL_FOLDER = '/content/drive/MyDrive/Intro to AI Final Project Files'
CNN_PATH    = f'{MODEL_FOLDER}/Pneumonia CNN Model.h5'
RESNET_PATH = f'{MODEL_FOLDER}/Pneumonia ResNet50 Model.h5'

# 👉 Path to dataset zip (in case it needs to be extracted)
ZIP_PATH = f'{MODEL_FOLDER}/final_dataset_80_10_10.zip'
EXTRACT_PATH = '/content/dataset'

# --- Helper: find folder anywhere in tree ---
def find_folder(root, target):
    for dirpath, dirnames, _ in os.walk(root):
        if target in dirnames:
            return os.path.join(dirpath, target)
    return None

# --- Extract dataset if needed ---
if not os.path.exists(EXTRACT_PATH) or not os.listdir(EXTRACT_PATH):
    print(f'📦 Extracting {ZIP_PATH}...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_PATH)
    print(f'✅ Extracted to {EXTRACT_PATH}')
else:
    print(f'✅ Dataset already at {EXTRACT_PATH}')

# --- Find the test folder ---
test_dir = find_folder(EXTRACT_PATH, 'test')
print(f'\n📁 test_dir: {test_dir}')
for cls in sorted(os.listdir(test_dir)):
    n = len(os.listdir(os.path.join(test_dir, cls)))
    print(f'  Test/{cls}: {n} images')

# --- Load models ---
print('\n📥 Loading Custom CNN...')
cnn = load_model(CNN_PATH)
print('✅ Custom CNN loaded')

print('📥 Loading ResNet50...')
resnet_model = load_model(RESNET_PATH)
print('✅ ResNet50 loaded')

# --- Set up test generators ---
test_datagen_cnn = ImageDataGenerator(rescale=1./255)
test_generator_cnn = test_datagen_cnn.flow_from_directory(
    test_dir, target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE, class_mode='binary',
    classes=classes, shuffle=False
)

test_datagen_resnet = ImageDataGenerator(preprocessing_function=resnet_preprocess)
test_generator_resnet = test_datagen_resnet.flow_from_directory(
    test_dir, target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE, class_mode='binary',
    classes=classes, shuffle=False
)

# --- Evaluate Custom CNN on TEST set ---
print('\n' + '='*60)
print('🧠 CUSTOM CNN — TEST SET RESULTS')
print('='*60)
cnn_test_loss, cnn_test_accuracy = cnn.evaluate(test_generator_cnn)
print(f'\n🎯 Custom CNN Test Loss:     {cnn_test_loss:.4f}')
print(f'🎯 Custom CNN Test Accuracy: {cnn_test_accuracy:.4f}')

test_generator_cnn.reset()
y_test_pred_probs_cnn = cnn.predict(test_generator_cnn)
y_test_pred_classes_cnn = [1 if prob > 0.5 else 0 for prob in y_test_pred_probs_cnn]
y_test_true = test_generator_cnn.classes

print('\nClassification Report (Custom CNN — Test):\n')
print(classification_report(y_test_true, y_test_pred_classes_cnn, target_names=classes))
cm_test_cnn = confusion_matrix(y_test_true, y_test_pred_classes_cnn)

# --- Evaluate ResNet50 on TEST set ---
print('\n' + '='*60)
print('🏗️ RESNET50 — TEST SET RESULTS')
print('='*60)
resnet_test_loss, resnet_test_accuracy = resnet_model.evaluate(test_generator_resnet)
print(f'\n🎯 ResNet50 Test Loss:     {resnet_test_loss:.4f}')
print(f'🎯 ResNet50 Test Accuracy: {resnet_test_accuracy:.4f}')

test_generator_resnet.reset()
y_test_pred_probs_resnet = resnet_model.predict(test_generator_resnet)
y_test_pred_classes_resnet = [1 if prob > 0.5 else 0 for prob in y_test_pred_probs_resnet]

print('\nClassification Report (ResNet50 — Test):\n')
print(classification_report(y_test_true, y_test_pred_classes_resnet, target_names=classes))
cm_test_resnet = confusion_matrix(y_test_true, y_test_pred_classes_resnet)

# --- Side-by-side TEST confusion matrices ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm_test_cnn, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes, ax=axes[0])
axes[0].set_title('Custom CNN — Test Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

sns.heatmap(cm_test_resnet, annot=True, fmt='d', cmap='Greens',
            xticklabels=classes, yticklabels=classes, ax=axes[1])
axes[1].set_title('ResNet50 — Test Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
plt.tight_layout()
plt.show()

# --- Final Test Comparison Table ---
test_results = {
    'Model': ['Custom CNN (from scratch)', 'ResNet50 (transfer learning)'],
    'Accuracy':  [accuracy_score(y_test_true, y_test_pred_classes_cnn),  accuracy_score(y_test_true, y_test_pred_classes_resnet)],
    'Precision': [precision_score(y_test_true, y_test_pred_classes_cnn), precision_score(y_test_true, y_test_pred_classes_resnet)],
    'Recall':    [recall_score(y_test_true, y_test_pred_classes_cnn),    recall_score(y_test_true, y_test_pred_classes_resnet)],
    'F1 Score':  [f1_score(y_test_true, y_test_pred_classes_cnn),        f1_score(y_test_true, y_test_pred_classes_resnet)]
}
test_results_df = pd.DataFrame(test_results)
print('\n📊 FINAL TEST SET COMPARISON 📊\n')
print(test_results_df.to_string(index=False))
test_results_df.to_csv('test_results_comparison.csv', index=False)

print('\n✅ All testing complete!')

<div style="border-radius:12px; padding: 20px; background-color: #d4edda; font-size:115%; text-align:left">

<h2 align="left"><font color=#155724>Summary 🎉</font></h2>

What this notebook covered:
1. Mounted Drive and unzipped the dataset to fast local Colab storage 📦
2. Built and trained **Custom CNN from scratch** (Coding Session 6 style) 🧠
3. Built and trained **ResNet50 with transfer learning** (Coding Session 7 style) 🏗️
4. Compared both models using accuracy, precision, recall, F1, and confusion matrices 📊
5. Saved both models for the testing teammate 🤝

**Files saved to your Drive:**
- `pneumonia_custom_cnn.h5` — the from-scratch CNN
- `pneumonia_resnet50.h5` — the transfer learning model
- `model_comparison.csv` — comparison table for slides

**For your presentation:** You now have everything you need for the Methodology and Results sections — pipeline diagrams, training curves, confusion matrices, and the comparison bar chart. ✨

</div>

In [ ]:
# Gradio demo - upload multiple X-rays and get predictions from both models
!pip install gradio -q

import gradio as gr
import numpy as np
import pandas as pd
import base64
from io import BytesIO
from PIL import Image
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess

# Settings
IMG_HEIGHT = 224
IMG_WIDTH = 224
classes = ['NORMAL', 'PNEUMONIA']

# Make sure models are loaded
try:
    cnn
    resnet_model
    print('Using in-memory models')
except NameError:
    MODEL_FOLDER = '/content/drive/MyDrive/Intro to AI Final Project Files'
    print('Loading models from Drive...')
    cnn = load_model(f'{MODEL_FOLDER}/Pneumonia CNN Model.h5')
    resnet_model = load_model(f'{MODEL_FOLDER}/Pneumonia ResNet50 Model.h5')
    print('Models loaded')


def image_to_base64(image, max_size=300):
    """Convert a PIL image to a base64 string for embedding in HTML."""
    display_img = image.copy()
    display_img.thumbnail((max_size, max_size), Image.LANCZOS)
    buffer = BytesIO()
    if display_img.mode != 'RGB':
        display_img = display_img.convert('RGB')
    display_img.save(buffer, format='JPEG', quality=85)
    img_str = base64.b64encode(buffer.getvalue()).decode()
    return f'data:image/jpeg;base64,{img_str}'


def predict_single(image):
    """Run both models on a single image."""
    image = image.convert('RGB').resize((IMG_WIDTH, IMG_HEIGHT))
    img_array = np.array(image)

    cnn_input = np.expand_dims(img_array / 255.0, axis=0)
    cnn_raw = float(cnn.predict(cnn_input, verbose=0)[0][0])
    cnn_label = classes[1] if cnn_raw > 0.5 else classes[0]

    resnet_input = np.expand_dims(resnet_preprocess(img_array.astype(np.float32)), axis=0)
    resnet_raw = float(resnet_model.predict(resnet_input, verbose=0)[0][0])
    resnet_label = classes[1] if resnet_raw > 0.5 else classes[0]

    return {
        'cnn_label': cnn_label,
        'cnn_normal': (1 - cnn_raw) * 100,
        'cnn_pneumonia': cnn_raw * 100,
        'resnet_label': resnet_label,
        'resnet_normal': (1 - resnet_raw) * 100,
        'resnet_pneumonia': resnet_raw * 100,
    }


def predict_xray_batch(files):
    """Takes multiple uploaded images and returns a styled HTML report."""
    if not files:
        return (
            '<div style="text-align:center; padding:40px; color:#333;">'
            '<h3 style="color:#333;">Upload chest X-ray images to see predictions</h3>'
            '<p style="color:#555;">Drag and drop one or more X-ray images above to begin analysis.</p>'
            '</div>',
            None,
            None
        )

    agree_count = 0
    disagree_count = 0
    high_conf_warnings = 0
    low_conf_warnings = 0

    cards_html = []
    table_rows = []

    for i, file_obj in enumerate(files, start=1):
        try:
            image = Image.open(file_obj.name if hasattr(file_obj, 'name') else file_obj)
        except Exception as e:
            cards_html.append(f'<div style="padding:15px; background:#fee; color:#721c24; border-radius:8px; margin-bottom:10px;">Image {i}: Could not load file ({e})</div>')
            continue

        filename = file_obj.name.split('/')[-1] if hasattr(file_obj, 'name') else f'image_{i}'
        img_base64 = image_to_base64(image)
        result = predict_single(image)

        # Status flags
        agree = result['cnn_label'] == result['resnet_label']
        if agree:
            agree_count += 1
        else:
            disagree_count += 1

        cnn_conf = max(result['cnn_normal'], result['cnn_pneumonia'])
        resnet_conf = max(result['resnet_normal'], result['resnet_pneumonia'])
        avg_conf = (cnn_conf + resnet_conf) / 2

        if avg_conf > 99:
            high_conf_warnings += 1
        if avg_conf < 70:
            low_conf_warnings += 1

        # Color coding for the prediction labels
        cnn_color = '#1b5e20' if result['cnn_label'] == 'NORMAL' else '#b71c1c'
        resnet_color = '#1b5e20' if result['resnet_label'] == 'NORMAL' else '#b71c1c'

        # Agreement banner - dark text on light backgrounds
        if agree:
            agree_banner = (
                f'<div style="background:#d4edda; color:#155724; padding:10px 15px; '
                f'border-radius:6px; border-left:4px solid #28a745; font-size:14px;">'
                f'<b style="color:#155724;">Both models agree:</b> '
                f'<span style="color:#155724; font-weight:600;">{result["cnn_label"]}</span>'
                f'</div>'
            )
        else:
            agree_banner = (
                f'<div style="background:#fff3cd; color:#664d03; padding:10px 15px; '
                f'border-radius:6px; border-left:4px solid #ffc107; font-size:14px;">'
                f'<b style="color:#664d03;">Models disagree</b> '
                f'<span style="color:#664d03;">&mdash; CNN says </span>'
                f'<span style="color:#664d03; font-weight:600;">{result["cnn_label"]}</span>'
                f'<span style="color:#664d03;">, ResNet50 says </span>'
                f'<span style="color:#664d03; font-weight:600;">{result["resnet_label"]}</span>'
                f'<span style="color:#664d03;">. This may be an edge case.</span>'
                f'</div>'
            )

        # Confidence banner - dark text on light backgrounds
        if avg_conf > 99:
            conf_banner = (
                f'<div style="background:#fff3cd; color:#664d03; padding:10px 15px; '
                f'border-radius:6px; margin-top:8px; border-left:4px solid #ffc107; font-size:14px;">'
                f'<b style="color:#664d03;">Suspiciously high confidence ({avg_conf:.1f}%)</b> '
                f'<span style="color:#664d03;">&mdash; check if this is an input the model was not trained on.</span>'
                f'</div>'
            )
        elif avg_conf < 70:
            conf_banner = (
                f'<div style="background:#f8d7da; color:#721c24; padding:10px 15px; '
                f'border-radius:6px; margin-top:8px; border-left:4px solid #dc3545; font-size:14px;">'
                f'<b style="color:#721c24;">Low confidence ({avg_conf:.1f}%)</b> '
                f'<span style="color:#721c24;">&mdash; borderline or uncertain prediction.</span>'
                f'</div>'
            )
        else:
            conf_banner = (
                f'<div style="background:#d1ecf1; color:#0c5460; padding:10px 15px; '
                f'border-radius:6px; margin-top:8px; border-left:4px solid #17a2b8; font-size:14px;">'
                f'<span style="color:#0c5460;">Normal confidence range ({avg_conf:.1f}%)</span>'
                f'</div>'
            )

        # Build the card
        card = f'''
        <div style="background:white; border:1px solid #e0e0e0; border-radius:12px; padding:20px; margin-bottom:16px; box-shadow:0 2px 4px rgba(0,0,0,0.05);">
            <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:14px; padding-bottom:12px; border-bottom:1px solid #f0f0f0;">
                <div style="font-weight:600; color:#222; font-size:15px;">Image {i}</div>
                <div style="color:#555; font-size:13px; font-family:monospace;">{filename}</div>
            </div>
            <div style="display:grid; grid-template-columns:200px 1fr; gap:20px; margin-bottom:14px; align-items:start;">
                <div style="background:#000; border-radius:8px; padding:8px; display:flex; align-items:center; justify-content:center; min-height:200px;">
                    <img src="{img_base64}" style="max-width:100%; max-height:280px; border-radius:4px; display:block;" alt="X-ray image"/>
                </div>
                <div style="display:grid; grid-template-columns:1fr 1fr; gap:12px;">
                    <div style="background:#f8f9fa; padding:14px; border-radius:8px; border-top:3px solid #1976d2;">
                        <div style="color:#444; font-size:12px; text-transform:uppercase; letter-spacing:0.5px; margin-bottom:6px; font-weight:600;">Custom CNN</div>
                        <div style="font-size:20px; font-weight:700; color:{cnn_color}; margin-bottom:10px;">{result["cnn_label"]}</div>
                        <div style="font-size:13px; color:#222; line-height:1.6;">
                            <span style="color:#444;">Normal:</span> <b style="color:#222;">{result["cnn_normal"]:.1f}%</b><br>
                            <span style="color:#444;">Pneumonia:</span> <b style="color:#222;">{result["cnn_pneumonia"]:.1f}%</b>
                        </div>
                    </div>
                    <div style="background:#f8f9fa; padding:14px; border-radius:8px; border-top:3px solid #388e3c;">
                        <div style="color:#444; font-size:12px; text-transform:uppercase; letter-spacing:0.5px; margin-bottom:6px; font-weight:600;">ResNet50</div>
                        <div style="font-size:20px; font-weight:700; color:{resnet_color}; margin-bottom:10px;">{result["resnet_label"]}</div>
                        <div style="font-size:13px; color:#222; line-height:1.6;">
                            <span style="color:#444;">Normal:</span> <b style="color:#222;">{result["resnet_normal"]:.1f}%</b><br>
                            <span style="color:#444;">Pneumonia:</span> <b style="color:#222;">{result["resnet_pneumonia"]:.1f}%</b>
                        </div>
                    </div>
                </div>
            </div>
            {agree_banner}
            {conf_banner}
        </div>
        '''
        cards_html.append(card)

        table_rows.append({
            'Image': filename,
            'Custom CNN': result['cnn_label'],
            'CNN Confidence': f'{cnn_conf:.1f}%',
            'ResNet50': result['resnet_label'],
            'ResNet Confidence': f'{resnet_conf:.1f}%',
            'Agreement': 'Yes' if agree else 'No'
        })

    # Summary card at the top - keep white text since background is dark gradient
    summary_html = f'''
    <div style="background:linear-gradient(135deg, #667eea 0%, #764ba2 100%); color:white; padding:24px; border-radius:12px; margin-bottom:20px; box-shadow:0 4px 12px rgba(0,0,0,0.1);">
        <div style="font-size:14px; text-transform:uppercase; letter-spacing:1px; opacity:0.95; margin-bottom:12px; color:white;">Batch Summary</div>
        <div style="display:grid; grid-template-columns:repeat(4, 1fr); gap:16px;">
            <div>
                <div style="font-size:32px; font-weight:700; color:white;">{len(files)}</div>
                <div style="font-size:12px; opacity:0.95; color:white;">Total images</div>
            </div>
            <div>
                <div style="font-size:32px; font-weight:700; color:white;">{agree_count}</div>
                <div style="font-size:12px; opacity:0.95; color:white;">Models agreed</div>
            </div>
            <div>
                <div style="font-size:32px; font-weight:700; color:white;">{disagree_count}</div>
                <div style="font-size:12px; opacity:0.95; color:white;">Models disagreed</div>
            </div>
            <div>
                <div style="font-size:32px; font-weight:700; color:white;">{high_conf_warnings + low_conf_warnings}</div>
                <div style="font-size:12px; opacity:0.95; color:white;">Flagged cases</div>
            </div>
        </div>
    </div>
    '''

    full_html = summary_html + ''.join(cards_html)

    df = pd.DataFrame(table_rows)

    # Disclaimer with dark text
    disclaimer_html = '''
    <div style="background:#fff3cd; color:#664d03; padding:14px 16px; border-radius:8px; margin-top:16px; border-left:4px solid #ffc107; font-size:13px;">
        <b style="color:#664d03;">Disclaimer:</b>
        <span style="color:#664d03;">This is a research demonstration only and should not be used for actual medical diagnosis. All predictions should be reviewed by a qualified radiologist.</span>
    </div>
    '''

    return full_html + disclaimer_html, df, df.to_csv(index=False)


# Custom CSS for the overall app
custom_css = """
.gradio-container {
    max-width: 1200px !important;
    margin: 0 auto !important;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif !important;
}
.app-header {
    text-align: center;
    padding: 30px 20px;
    background: linear-gradient(135deg, #1976d2 0%, #388e3c 100%);
    border-radius: 12px;
    margin-bottom: 24px;
    color: white;
}
.app-header h1 {
    font-size: 32px !important;
    font-weight: 700 !important;
    margin: 0 0 8px 0 !important;
    color: white !important;
}
.app-header p {
    font-size: 15px !important;
    opacity: 0.95 !important;
    margin: 0 !important;
    color: white !important;
}
.section-label {
    font-size: 14px !important;
    font-weight: 600 !important;
    color: #333 !important;
    text-transform: uppercase;
    letter-spacing: 0.5px;
    margin-bottom: 8px !important;
}
"""


# Build the interface using Blocks for better layout control
with gr.Blocks(css=custom_css, theme=gr.themes.Soft(primary_hue='blue', secondary_hue='green')) as demo:

    gr.HTML('''
        <div class="app-header">
            <h1>Chest X-Ray Pneumonia Classifier</h1>
            <p>Comparing a custom CNN vs ResNet50 transfer learning on chest X-ray images</p>
            <p style="font-size:13px; opacity:0.9; margin-top:8px;">CAP 4630 Final Project</p>
        </div>
    ''')

    with gr.Row():
        with gr.Column(scale=1):
            gr.HTML('<div class="section-label">Upload X-rays</div>')
            file_input = gr.File(
                file_count='multiple',
                file_types=['image'],
                label='',
                height=200
            )
            with gr.Row():
                analyze_btn = gr.Button('Analyze Images', variant='primary', size='lg')
                clear_btn = gr.Button('Clear', size='lg')

            gr.HTML('''
                <div style="background:#e3f2fd; padding:14px; border-radius:8px; margin-top:12px; font-size:13px; color:#0d47a1; border-left:4px solid #1976d2;">
                    <b style="color:#0d47a1;">Tip for edge cases:</b><br>
                    <span style="color:#0d47a1;">Try uploading X-rays the model was not trained on (lung cancer, tuberculosis) or non-medical images to see how the system handles out-of-distribution inputs.</span>
                </div>
            ''')

        with gr.Column(scale=2):
            gr.HTML('<div class="section-label">Prediction Results</div>')
            output_html = gr.HTML(value=(
                '<div style="text-align:center; padding:60px 20px; color:#444; background:#f8f9fa; border-radius:12px; border:2px dashed #dee2e6;">'
                '<h3 style="color:#222; margin:0 0 8px 0;">No images analyzed yet</h3>'
                '<p style="margin:0; color:#444;">Upload chest X-ray images and click <b style="color:#222;">Analyze Images</b> to see predictions.</p>'
                '</div>'
            ))

    with gr.Accordion('Results Table (downloadable)', open=False):
        output_table = gr.Dataframe(
            headers=['Image', 'Custom CNN', 'CNN Confidence', 'ResNet50', 'ResNet Confidence', 'Agreement'],
            interactive=False,
            wrap=True
        )
        output_csv = gr.Textbox(
            label='CSV (copy and paste into a spreadsheet)',
            lines=4,
            visible=True
        )

    with gr.Accordion('About this project', open=False):
        gr.Markdown('''
        **Models compared:**
        - **Custom CNN** built from scratch (Sequential architecture with Conv2D + MaxPooling layers, similar to Coding Session 6)
        - **ResNet50** with transfer learning (pre-trained on ImageNet, similar to Coding Session 7)

        **Dataset:** Chest X-Ray Pneumonia Balanced Dataset from Kaggle &mdash; split 80% train / 10% validation / 10% test.

        **How predictions work:** Each image is resized to 224x224 and passed through both models. The Custom CNN expects pixel values rescaled to 0-1, while ResNet50 uses its own preprocessing function. Each model outputs a probability between 0 and 1, where values above 0.5 indicate PNEUMONIA.

        **About the warnings:**
        - **Suspiciously high confidence (>99%)** can indicate an out-of-distribution input that the model is being forced to classify into one of its two known classes.
        - **Low confidence (<70%)** suggests a borderline or uncertain prediction that warrants careful review.
        - **Model disagreement** between CNN and ResNet50 also flags potentially difficult cases.

        **Disclaimer:** This is a research demonstration for an academic project. It should not be used for actual medical diagnosis under any circumstances.
        ''')

    # Wire up the buttons
    analyze_btn.click(
        fn=predict_xray_batch,
        inputs=[file_input],
        outputs=[output_html, output_table, output_csv]
    )

    def clear_all():
        return (
            None,
            (
                '<div style="text-align:center; padding:60px 20px; color:#444; background:#f8f9fa; border-radius:12px; border:2px dashed #dee2e6;">'
                '<h3 style="color:#222; margin:0 0 8px 0;">No images analyzed yet</h3>'
                '<p style="margin:0; color:#444;">Upload chest X-ray images and click <b style="color:#222;">Analyze Images</b> to see predictions.</p>'
                '</div>'
            ),
            None,
            ''
        )

    clear_btn.click(
        fn=clear_all,
        outputs=[file_input, output_html, output_table, output_csv]
    )


# Launch with public link
demo.launch(share=True)

Using in-memory models


/tmp/ipykernel_3576/1480891887.py:288: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Soft(primary_hue='blue', secondary_hue='green')) as demo:
/tmp/ipykernel_3576/1480891887.py:288: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Soft(primary_hue='blue', secondary_hue='green')) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://af14ab2f5a4feda352.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os

folder = '/content/drive/MyDrive/Intro to AI Final Project Files'

if os.path.exists(folder):
    print(f'Folder exists. Contents:')
    for f in sorted(os.listdir(folder)):
        full = os.path.join(folder, f)
        size_mb = os.path.getsize(full) / (1024 * 1024) if os.path.isfile(full) else 0
        print(f'  {f}  ({size_mb:.1f} MB)' if size_mb > 0 else f'  {f}/')
else:
    print(f'Folder does NOT exist at: {folder}')
    print('\nWhat IS in MyDrive root:')
    for f in sorted(os.listdir('/content/drive/MyDrive'))[:30]:
        print(f'  {f}')